In [1]:
#from pyspark.sql import SparkSession

# Initialize Spark session
#spark = SparkSession.builder.appName("Spark-RDD-demo").getOrCreate()

## Use SparkContext instead of SparkSession

SparkContext is the core entry point for Spark's RDD API and is responsible for:

* connecting with Spark cluster
* allocating resources (CPU, memory)
* coordinate execution 


In [66]:
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

## Get an RDD from a file

In [67]:
rdd = rdd1 = sc.textFile("data/file.txt")

In [68]:
rdd

data/file.txt MapPartitionsRDD[45] at textFile at NativeMethodAccessorImpl.java:0

Prints something like

`data/file.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0`

What does it mean:

* `MapPartitionsRDD`: indicates that data is processed partition by partition
* `MapPartitionsRDD[1]` refers to the partition index
* `textFile` says it recognized a text content file
* `NativeMethodAccessorImpl.java:0` indicates a native Java function was called



In [69]:
rdd.collect()

['hello world', 'hello spark', 'big data spark']

In [70]:
rdd.first()

'hello world'

In [71]:
# dir(rdd)

In [72]:
rdd.count()

3

In [73]:
for rec in rdd.take(2):  # iterate over specific number of lines
    print(rec)

hello world
hello spark


In [74]:
rdd.toLocalIterator()  # get a generator to loop over rdd

<generator object _local_iterator_from_socket.<locals>.PyLocalIterable.__iter__ at 0x7f079d703140>

## Get an RDD from a Python collection

We can configureThe way partitions are

In [75]:
rdd = rdd2 = sc.parallelize(list(range(111)))

In [76]:
rdd

ParallelCollectionRDD[49] at readRDDFromFile at PythonRDD.scala:289

In [77]:
rdd.getNumPartitions()

16

In [78]:
rdd.glom().collect()  # glom: group elements per partition into list

[[0, 1, 2, 3, 4, 5],
 [6, 7, 8, 9, 10, 11],
 [12, 13, 14, 15, 16, 17],
 [18, 19, 20, 21, 22, 23],
 [24, 25, 26, 27, 28, 29],
 [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41],
 [42, 43, 44, 45, 46, 47],
 [48, 49, 50, 51, 52, 53],
 [54, 55, 56, 57, 58, 59],
 [60, 61, 62, 63, 64, 65],
 [66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77],
 [78, 79, 80, 81, 82, 83],
 [84, 85, 86, 87, 88, 89],
 [90, 91, 92, 93, 94, 95],
 [96, 97, 98, 99, 100, 101],
 [102, 103, 104, 105, 106, 107, 108, 109, 110]]

## Create an RDD from another RDD via transformation

In [79]:
rdd3 = rdd2.filter(lambda x: x % 4 == 0)
rdd3

PythonRDD[51] at RDD at PythonRDD.scala:53

In [80]:
rdd3.collect()

[0,
 4,
 8,
 12,
 16,
 20,
 24,
 28,
 32,
 36,
 40,
 44,
 48,
 52,
 56,
 60,
 64,
 68,
 72,
 76,
 80,
 84,
 88,
 92,
 96,
 100,
 104,
 108]

In [81]:
rdd3.glom().collect()  # glom: group elements per partition into list

[[0, 4],
 [8],
 [12, 16],
 [20],
 [24, 28],
 [32, 36, 40],
 [44],
 [48, 52],
 [56],
 [60, 64],
 [68, 72, 76],
 [80],
 [84, 88],
 [92],
 [96, 100],
 [104, 108]]

## RDD lineage

In [82]:
print(rdd1.toDebugString().decode())

(2) data/file.txt MapPartitionsRDD[45] at textFile at NativeMethodAccessorImpl.java:0 []
 |  data/file.txt HadoopRDD[44] at textFile at NativeMethodAccessorImpl.java:0 []


In [83]:
print(rdd2.toDebugString().decode())

(16) ParallelCollectionRDD[49] at readRDDFromFile at PythonRDD.scala:289 []


In [84]:
print(rdd3.toDebugString().decode())

(16) PythonRDD[51] at RDD at PythonRDD.scala:53 []
 |   ParallelCollectionRDD[49] at readRDDFromFile at PythonRDD.scala:289 []


## Transformations part 1: map, flatMap

In [85]:
rdd = rdd1

In [86]:
strings = rdd.map(lambda x: x)
strings.collect()

['hello world', 'hello spark', 'big data spark']

In [87]:
strings

PythonRDD[53] at collect at /tmp/ipykernel_7506/3950525423.py:2

In [88]:
strings = rdd.map(lambda x: x.split())
strings.collect()

[['hello', 'world'], ['hello', 'spark'], ['big', 'data', 'spark']]

In [89]:
words = rdd.flatMap(lambda x: x.split())
words.collect()

['hello', 'world', 'hello', 'spark', 'big', 'data', 'spark']

In [90]:
upperwords = rdd.flatMap(lambda x: x.upper().split())
upperwords.collect()

['HELLO', 'WORLD', 'HELLO', 'SPARK', 'BIG', 'DATA', 'SPARK']

In [91]:
chars = rdd.flatMap(lambda x: x)
chars.collect()[:15]

['h', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd', 'h', 'e', 'l', 'l']

In [92]:
print(words.toDebugString().decode())

(2) PythonRDD[55] at collect at /tmp/ipykernel_7506/3141199366.py:2 []
 |  data/file.txt MapPartitionsRDD[45] at textFile at NativeMethodAccessorImpl.java:0 []
 |  data/file.txt HadoopRDD[44] at textFile at NativeMethodAccessorImpl.java:0 []


In [93]:
# list of lists
pairs = words.map(lambda w: (w, 1))
pairs.collect()

[('hello', 1),
 ('world', 1),
 ('hello', 1),
 ('spark', 1),
 ('big', 1),
 ('data', 1),
 ('spark', 1)]

In [94]:
help(pairs.reduceByKey)

Help on method reduceByKey in module pyspark.rdd:

reduceByKey(func: Callable[[~V, ~V], ~V], numPartitions: Optional[int] = None, partitionFunc: Callable[[~K], int] = <function portable_hash at 0x7f07bc6fd3a0>) -> 'RDD[Tuple[K, V]]' method of pyspark.rdd.PipelinedRDD instance
    Merge the values for each key using an associative and commutative reduce function.
    
    This will also perform the merging locally on each mapper before
    sending results to a reducer, similarly to a "combiner" in MapReduce.
    
    Output will be partitioned with `numPartitions` partitions, or
    the default parallelism level if `numPartitions` is not specified.
    Default partitioner is hash-partition.
    
    .. versionadded:: 1.6.0
    
    Parameters
    ----------
    func : function
        the reduce function
    numPartitions : int, optional
        the number of partitions in new :class:`RDD`
    partitionFunc : function, optional, default `portable_hash`
        function to compute the pa

In [57]:
# reduceByKey takes a (key,value) RDD, uses the keys to build bags and applies the
# provided function to the already stored value per key (just stores the first value)
# in the lamba below a refers to the already reduced stored value and b refers
# to the next value encountered
wordCounts = pairs.reduceByKey(lambda a, b: a + b)
wordCounts.collect()

[('hello', 2), ('world', 1), ('big', 1), ('spark', 2), ('data', 1)]

In [62]:
print(wordCounts.toDebugString().decode())

(2) PythonRDD[43] at collect at /tmp/ipykernel_7506/2865902819.py:2 []
 |  MapPartitionsRDD[42] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[41] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[40] at reduceByKey at /tmp/ipykernel_7506/2865902819.py:1 []
    |  PythonRDD[39] at reduceByKey at /tmp/ipykernel_7506/2865902819.py:1 []
    |  data/file.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  data/file.txt HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


## Transformations part 2: filter

In [134]:
rdd = sc.parallelize([1, 2, 3, 4, 5, 6])
even_rdd = rdd.filter(lambda x: x % 2 == 0)
even_rdd.collect()

[2, 4, 6]

## Caching

Without caching RDD will repeat the whole lineage graph for each action verb from the top.

In [107]:
rdd = sc.parallelize(list(range(150)))
rdd_uncached = rdd.map(lambda x: x * 2)
rdd_cached = rdd.map(lambda x: x * 2).cache()
print(rdd_cached.count())   # triggers computation and caches result
print(rdd_cached.collect()) # uses cached data

150
[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126, 128, 130, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 152, 154, 156, 158, 160, 162, 164, 166, 168, 170, 172, 174, 176, 178, 180, 182, 184, 186, 188, 190, 192, 194, 196, 198, 200, 202, 204, 206, 208, 210, 212, 214, 216, 218, 220, 222, 224, 226, 228, 230, 232, 234, 236, 238, 240, 242, 244, 246, 248, 250, 252, 254, 256, 258, 260, 262, 264, 266, 268, 270, 272, 274, 276, 278, 280, 282, 284, 286, 288, 290, 292, 294, 296, 298]


In [108]:
rdd_uncached.is_cached, rdd_cached.is_cached, rdd1.is_cached

(False, True, False)

In [110]:
print(rdd_uncached.toDebugString().decode())

(16) PythonRDD[71] at RDD at PythonRDD.scala:53 []
 |   ParallelCollectionRDD[68] at readRDDFromFile at PythonRDD.scala:289 []


In [111]:
print(rdd_cached.toDebugString().decode())

(16) PythonRDD[69] at RDD at PythonRDD.scala:53 [Memory Serialized 1x Replicated]
 |        CachedPartitions: 16; MemorySize: 1788.0 B; DiskSize: 0.0 B
 |   ParallelCollectionRDD[68] at readRDDFromFile at PythonRDD.scala:289 [Memory Serialized 1x Replicated]


In [117]:
help(rdd_uncached.getStorageLevel)
help(rdd_uncached.getStorageLevel())

Help on method getStorageLevel in module pyspark.rdd:

getStorageLevel() -> pyspark.storagelevel.StorageLevel method of pyspark.rdd.PipelinedRDD instance
    Get the RDD's current storage level.
    
    .. versionadded:: 1.0.0
    
    Returns
    -------
    :class:`StorageLevel`
        current :class:`StorageLevel`
    
    See Also
    --------
    :meth:`RDD.name`
    
    Examples
    --------
    >>> rdd = sc.parallelize([1,2])
    >>> rdd.getStorageLevel()
    StorageLevel(False, False, False, False, 1)
    >>> print(rdd.getStorageLevel())
    Serialized 1x Replicated

Help on StorageLevel in module pyspark.storagelevel object:

class StorageLevel(builtins.object)
 |  StorageLevel(useDisk: bool, useMemory: bool, useOffHeap: bool, deserialized: bool, replication: int = 1)
 |  
 |  Flags for controlling the storage of an RDD. Each StorageLevel records whether to use memory,
 |  whether to drop the RDD to disk if it falls out of memory, whether to keep the data in memory
 |  in a

In [115]:
# StorageLevel(useDisk, useMemory, useofHeap, deserialized, replication)

In [112]:
rdd_uncached.getStorageLevel()

StorageLevel(False, False, False, False, 1)

In [113]:
rdd_cached.getStorageLevel()

StorageLevel(False, True, False, False, 1)

## Creating co-groups (not actual joins!)

It 'stupidly' associates all data belonging to a certain key. To add logic we need join.

In [133]:
# random example
scores_rdd = sc.parallelize([("Alice",85),("Bob",90),("Alice",95)])
courses_rdd = sc.parallelize([("Alice","Math"),("Bob","Science"),("Charlie","History")])

result = scores_rdd.cogroup(courses_rdd)

for student, (scores, courses) in result.collect():
    print(student, list(scores), list(courses))

Charlie [] ['History']
Alice [85, 95] ['Math']
Bob [90] ['Science']


In [126]:
result.collect()

[('Charlie',
  (<pyspark.resultiterable.ResultIterable at 0x7f0745317c10>,
   <pyspark.resultiterable.ResultIterable at 0x7f079d394bd0>)),
 ('Alice',
  (<pyspark.resultiterable.ResultIterable at 0x7f079d396b50>,
   <pyspark.resultiterable.ResultIterable at 0x7f079d396c10>)),
 ('Bob',
  (<pyspark.resultiterable.ResultIterable at 0x7f079d3d6cd0>,
   <pyspark.resultiterable.ResultIterable at 0x7f079d397e50>))]

In [122]:
print(result.toDebugString().decode())

(32) PythonRDD[111] at collect at /tmp/ipykernel_7506/1734334852.py:6 []
 |   MapPartitionsRDD[110] at mapPartitions at PythonRDD.scala:160 []
 |   ShuffledRDD[109] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(32) PairwiseRDD[108] at cogroup at /tmp/ipykernel_7506/1734334852.py:4 []
    |   PythonRDD[107] at cogroup at /tmp/ipykernel_7506/1734334852.py:4 []
    |   UnionRDD[106] at union at NativeMethodAccessorImpl.java:0 []
    |   PythonRDD[104] at RDD at PythonRDD.scala:53 []
    |   ParallelCollectionRDD[102] at readRDDFromFile at PythonRDD.scala:289 []
    |   PythonRDD[105] at RDD at PythonRDD.scala:53 []
    |   ParallelCollectionRDD[103] at readRDDFromFile at PythonRDD.scala:289 []


## Demo ETL

In [138]:
# SparkContext
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

# Extract
rdd = sc.textFile("data/file.txt")

# Transform
clean_rdd = rdd.filter(lambda line: line.strip() != "")
word_counts = (clean_rdd
               .flatMap(lambda line: line.split(" "))
               .map(lambda word: (word, 1))
               .reduceByKey(lambda a, b: a + b))

# Check partitions
print("Partitions in word_counts RDD:", word_counts.getNumPartitions())

# Load
word_counts.saveAsTextFile("output/")  # one output file per partition


Partitions in word_counts RDD: 2


In [207]:

# Cleanup (optional)
from pyspark.sql import SparkSession

# Stop existing Spark session if running
try:
    spark.stop()
except:
    pass

In [196]:
from pyspark.sql import SparkSession

# Create SparkSession FIRST with configuration
spark = (SparkSession.builder
         .appName("MyApp")
         .config("spark.default.parallelism", 4)  # max no of tasks Spark can run in parallel
         .getOrCreate())

# Get SparkContext from SparkSession
sc = spark.sparkContext

# Create RDD (if no number of partitions specified → uses sc.defaultParallelism = 16 )
# Note: make sure numPartitions >= parallelism to utilize all cores allocated
rdd = sc.parallelize(
    list(range(24)),
    #4,  # number of partitions
)

# Check number of partitions
print("Number of partitions:", rdd.getNumPartitions())

# See distribution of data
print("Elements per partition:", rdd.glom().collect())


Number of partitions: 16
Elements per partition: [[0], [1, 2], [3], [4, 5], [6], [7, 8], [9], [10, 11], [12], [13, 14], [15], [16, 17], [18], [19, 20], [21], [22, 23]]


In [7]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass
    
# Create SparkSession FIRST with configuration
spark = (SparkSession.builder
         .appName("MyApp")
         .config("spark.default.parallelism", "4")  # max no of tasks Spark can run in parallel
         .getOrCreate())
print(
    "Initial config:\n  ",
    "SparkSession parallelism = ", spark.conf._jconf.get("spark.default.parallelism"),
    "SparkContext.defaultParallelism =", spark.sparkContext.defaultParallelism,
)

spark.conf.set("spark.default.parallelism", 6)
print(
    "Trying session re-config:\n  ",
    "SparkSession parallelism = ", spark.conf._jconf.get("spark.default.parallelism"),
    "SparkContext.defaultParallelism =", spark.sparkContext.defaultParallelism,
)

try:
    spark.stop()
    pass
except:
    pass
    
spark = (SparkSession.builder
         .appName("OtherApp")
         .config("spark.default.parallelism", "2")
         .getOrCreate())

print(
    "New initial config after stopping session:\n  ",
    "SparkSession parallelism = ", spark.conf._jconf.get("spark.default.parallelism"),
    "SparkContext.defaultParallelism =", spark.sparkContext.defaultParallelism,
)


Initial config:
   SparkSession parallelism =  4 SparkContext.defaultParallelism = 4
Trying session re-config:
   SparkSession parallelism =  6 SparkContext.defaultParallelism = 4
New initial config after stopping session:
   SparkSession parallelism =  2 SparkContext.defaultParallelism = 2


In [8]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass
    
# Create SparkSession FIRST with configuration
spark = (SparkSession.builder
         .appName("MyApp")
         .config("spark.default.parallelism", "4")  # max no of tasks Spark can run in parallel
         .getOrCreate())
print(
    "Initial config:\n  ",
    "SparkSession parallelism = ", spark.conf._jconf.get("spark.default.parallelism"),
    "SparkContext.defaultParallelism =", spark.sparkContext.defaultParallelism,
)

spark.conf.set("spark.default.parallelism", 6)
print(
    "Trying session re-config:\n  ",
    "SparkSession parallelism = ", spark.conf._jconf.get("spark.default.parallelism"),
    "SparkContext.defaultParallelism =", spark.sparkContext.defaultParallelism,
)

try:
    spark.stop()
except:
    pass
    
spark = (SparkSession.builder
         .appName("OtherApp")
         .config("spark.default.parallelism", "2")
         .getOrCreate())
print(
    "New initial config after stopping session:\n  ",
    "SparkSession parallelism = ", spark.conf._jconf.get("spark.default.parallelism"),
    "SparkContext.defaultParallelism =", spark.sparkContext.defaultParallelism,
)

# Get SparkContext from SparkSession
sc = spark.sparkContext

# Create RDD (if no number of partitions specified → uses sc.defaultParallelism = 16 )
# Note: make sure numPartitions >= parallelism to utilize all cores allocated
rdd = sc.parallelize(
    list(range(24)),
    #4,  # number of partitions
)

# Check number of partitions
print("Number of partitions =", rdd.getNumPartitions())

# See distribution of data
print("Elements per partition:", rdd.glom().collect())

print()

Initial config:
   SparkSession parallelism =  4 SparkContext.defaultParallelism = 4
Trying session re-config:
   SparkSession parallelism =  6 SparkContext.defaultParallelism = 4
New initial config after stopping session:
   SparkSession parallelism =  2 SparkContext.defaultParallelism = 2
Number of partitions = 2
Elements per partition: [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]]



In [257]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

try:
    spark.SparkContext.stop()
except:
    pass
try:
    spark.stop()
except:
    pass
    
# Create SparkSession FIRST with configuration
spark = (SparkSession.builder
         .appName("MyApp")
         .config("spark.default.parallelism", "4")  # max no of tasks Spark can run in parallel
         .getOrCreate())
sc = spark.sparkContext
conf = sc.getConf()  # gets a copy
conf.set("spark.default.parallelism", 2)
sc.stop()


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: org.apache.spark.SparkException: Only one SparkContext should be running in this JVM (see SPARK-2243).The currently running SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
jdk.internal.reflect.GeneratedConstructorAccessor81.newInstance(Unknown Source)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:499)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:480)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)


In [246]:
print(
    spark.conf._jconf.get("spark.default.parallelism"),
    spark.sparkContext.defaultParallelism,
)

4 4


In [238]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass
    
# Create SparkSession FIRST with configuration
spark = (SparkSession.builder
         .appName("MyApp")
         .config("spark.default.parallelism", "4")  # max no of tasks Spark can run in parallel
         .getOrCreate())
print(
    spark.conf._jconf.get("spark.default.parallelism"),
    spark.sparkContext.defaultParallelism,
)

spark.conf.set("spark.default.parallelism", 6)
print(
    spark.conf._jconf.get("spark.default.parallelism"),
    spark.sparkContext.defaultParallelism,
)

spark.conf.set("spark.default.parallelism", "8")
print(
    spark.conf._jconf.get("spark.default.parallelism"),
    spark.sparkContext.defaultParallelism,
)

spark = (SparkSession.builder
         .appName("OtherApp")
         .config("spark.default.parallelism", "2")
         .getOrCreate())
print(
    spark.conf._jconf.get("spark.default.parallelism"),
    spark.sparkContext.defaultParallelism,
)


4 4
6 4
8 4
2 4


## How to inspect Spark configuration

'2'

In [184]:
spark.conf._jconf.getAll()

JavaObject id=o1559

### Manual parsing

In [234]:
print('\n\n'.join(
    spark.conf._jconf.getAll().toString()
    .split(',')
))

Map(spark.sql.warehouse.dir -> file:/home/jovyan/work/spark-warehouse

 spark.executor.extraJavaOptions -> -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false

 spark.driver.host -> 5cffd8b06359

 spark.serializer.objectStreamReset -> 100

 spark.

### Convert to dict

In [187]:
scala_map = spark.conf._jconf.getAll()
python_dict = {}

iterator = scala_map.iterator()
while iterator.hasNext():
    entry = iterator.next()
    key = entry._1()  # Scala tuple's first element (key)
    value = entry._2()  # Scala tuple's second element (value)
    python_dict[key] = value

from pprint import pprint
pprint(python_dict)


{'spark.app.id': 'local-1777540198450',
 'spark.app.name': 'MyApp',
 'spark.app.startTime': '1777540197792',
 'spark.app.submitTime': '1777540197599',
 'spark.default.parallelism': '4',
 'spark.driver.extraJavaOptions': '-Djava.net.preferIPv6Addresses=false '
                                  '-XX:+IgnoreUnrecognizedVMOptions '
                                  '--add-opens=java.base/java.lang=ALL-UNNAMED '
                                  '--add-opens=java.base/java.lang.invoke=ALL-UNNAMED '
                                  '--add-opens=java.base/java.lang.reflect=ALL-UNNAMED '
                                  '--add-opens=java.base/java.io=ALL-UNNAMED '
                                  '--add-opens=java.base/java.net=ALL-UNNAMED '
                                  '--add-opens=java.base/java.nio=ALL-UNNAMED '
                                  '--add-opens=java.base/java.util=ALL-UNNAMED '
                                  '--add-opens=java.base/java.util.concurrent=ALL-UNNAMED '


In [185]:
# dir(spark.conf._jconf.getAll())

In [190]:
sc.getConf().getAll()

[('spark.driver.port', '44527'),
 ('spark.executor.id', 'driver'),
 ('spark.app.name', 'pyspark-shell'),
 ('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'),
 ('spark.app.id', 'local-1777540198450'),
 ('spar

In [ ]:

# Stop existing Spark session if running
try:
    spark.stop()
except:
    pass